# relay_mech

> Phase 23 (`specs/multiagent/phase23/`, experiment C1b): the measures of a relay board. `relayColumns` (FR-D1): relays confirmed on arrival, the bridge's filter gain, re-relays, a dedicated bridge's opens (`roleBreaks`), contamination on coupled subtasks alone, and the two ceilings, `truthConfirmable` and `liarConfirmed`. `c1bColumns` is C1's columns plus these; `runC1bCell` runs a C1b cell with them, numpy or Claude; `reusedRows23` replays C1's boards with them from the cache at $0, every saved column unchanged.

In [ ]:
#| default_exp relay_mech

In [ ]:
#| hide
from nbdev.showdoc import *

A relay's view of the mirror is the posts on that subtask's clue 1 visible to the bridge when it acted (`colony_trace._before`: earlier, or at the same time by a lower slot). `contamination` itself is `traceBoard`'s, which counts lies that arrive by relay or window through `falseValues`' hook (FR-D1); `coupledContamination` is the same count over honest answers on coupled subtasks only. C1's `relaysUsed` already counts relays whose value entered a correct answer, so no second column is added for it.

In [ ]:
#| export
import numpy as np

from explRL.bridge_mech import c1Columns
from explRL.clue_colony import Q
from explRL.colony_mech import NAN, _diff, _key
from explRL.colony_trace import _before, contaminatedBy, falseValues, replay
from explRL.llm_runs import LLMConfig, cellRow, runLLMCell

RELAY_COLS = ('arrivalConfirmed', 'filterGain', 'reRelays', 'roleBreaks', 'coupledContamination', 'truthConfirmable', 'liarConfirmed')
_mean = lambda xs: float(np.mean(xs)) if len(xs) else NAN


def _seen(ml, k, t, i):
    "{r: senders} of clue 1's mirror posts on k visible to agent i acting at t."
    out = {}
    for x in ml:
        if x[2] == k and x[3] == 1 and _before(x[:2], t, i):
            out.setdefault(x[4], set()).add(x[1])
    return out


def relayColumns(cfg, board, answers):
    """FR-D1's new columns of one board (NaN where a board has no relays, dedicated bridges or coupled subtasks). answers:
    {(t, agent): answer}, as c1Columns reads them."""
    nl, b = cfg.nLiar, board
    ml, rl, cp = getattr(b, 'mirrorLog', []), getattr(b, 'relayLog', []), getattr(b, 'coupled', {})
    true = lambda k, r: b.tasks[k].task.clue(1)[1] == r
    conf, avail, wrong, last, rerel = [], [], [], {}, 0
    for t, i, k, _, r, _ in rl:
        m = _seen(ml, k, t, i)
        conf.append(len(m.get(r, ())) >= Q)
        wrong.append(not true(k, r))
        if m:
            avail.append(np.mean([not true(k, v) for v in m]))
            top = max(len(s) for s in m.values())
            rerel += k in last and last[k] != r and len(m.get(last[k], ())) < top and len(m.get(r, ())) == top
        last[k] = r
    fv, rb = falseValues(b, True), getattr(b, 'roleBridges', frozenset())
    subs = [s for s in b.submits if s[1] >= nl and s[2] in cp]
    mir = {k: {} for k in cp}
    for x in ml:
        if x[3] == 1 and x[2] in mir:
            mir[x[2]].setdefault(x[4], set()).add(x[1])
    return dict(arrivalConfirmed=_mean(conf), filterGain=_mean(avail) - _mean(wrong) if avail else NAN, reRelays=float(rerel) if rl else NAN,
                roleBreaks=float(sum(row[0] in rb and row[3] == 'travel' for row in b.ledger)) if rb else NAN,
                coupledContamination=_mean([bool(contaminatedBy(b, s, answers.get((float(s[0]), s[1])), fv)) for s in subs]),
                truthConfirmable=_mean([len(v.get(b.tasks[k].task.clue(1)[1], ())) >= Q for k, v in mir.items()]),
                liarConfirmed=_mean([any(not true(k, r) and sum(a < nl for a in s) >= 2 for r, s in v.items()) for k, v in mir.items()]))


def c1bColumns(cfg, rp):
    "Every C1b row's columns from a replay: c1Columns (stage4Columns, bridgeColumns, contestColumns) and relayColumns."
    return {**c1Columns(cfg, rp), **relayColumns(cfg, rp.board, rp.answers)}


def runC1b(cfg, seed, client):
    "runClue's signature for runLLMCell: the replay's row with C1b's columns, and its turn log."
    rp = replay(cfg, seed, client)
    return [c1bColumns(cfg, rp)], rp.log


def runC1bCell(cell):
    "For runWorkload (picklable), as runC1Cell: a C1b cell with C1b's columns; numpy cells through `replay`."
    cfg = LLMConfig.fromDict(cell['cfg'])
    if cell['runner'] == 'numpy':
        return [cellRow(cell, cfg, c1bColumns(cfg, replay(cfg, cell['seed'])))]
    return runLLMCell(cell, run=runC1b)


def reusedRows23(client, prior, cells):
    """C1's boards replayed through client(cfg) (the cache-only client: $0) with C1b's columns (FR-C2); every column of the
    matching prior row (same study, level, N and seed: C1's runs.csv and reused.csv) must be unchanged."""
    idx, out = {_key(r): r for r in prior}, []
    for c in cells:
        cfg = LLMConfig.fromDict(c['cfg'])
        row = cellRow(c, cfg, c1bColumns(cfg, replay(cfg, c['seed'], client(cfg))))
        bad = _diff(idx[_key(c)], row)
        assert not bad, f"replay of {_key(c)} differs from its saved row in {bad[:5]}"
        out.append(row)
    return out

In [ ]:
# test_relay_columns (FR-D1, AC-5's count): on planted moves each column has its definition; a dedicated bridge's opens are
# roleBreaks; on a numpy bridgeFix board every column is present and in range, and C1's columns are unchanged by the hook
from explRL.clue_agents import ClueBoard
from explRL.colony_grid import WORLD18
from explRL.relay_board import RelayBoardEnv
from explRL.relay_grid import c1bCell
kw, G = dict(WORLD18), range(16, 32)
cfg = LLMConfig(task='clue', channel='blackboard', N=64, turns=40, nLiar=16, memory='notes',
                world=dict(kw, colony=True, guards=16, squads=8, couple=0.25, bridges=1, evidence=True, bridgeRole=True))
b, ch = RelayBoardEnv(1800, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, bridges=1, evidence=True, bridgeRole=True, **kw), ClueBoard()
k = next(k for k in b.open if k in b.coupled and min(b.tasks[k].holders[1]) >= 16)
s, s2 = b.coupled[k]
(p1, r1), br = b.tasks[k].task.clue(1), b.bridges[s][0]
h1, h2 = sorted(b.tasks[k].holders[1])
l1, l2 = [i for i in range(16) if i % 8 == s2]
lie = (r1 + 1) % p1
for t, i, r in ((1, l1, lie), (1, l2, lie)):                        # two liars confirm a lie on the mirror
    b.apply(i, ('share', (k, 1, r)), t)
b.apply(br, ('share', (k, 1, lie)), 2)                                # relay 1: confirmed on arrival, false; mirror false share 1
for t, i in ((3, h1), (3, h2), (4, next(x for x in range(32, 64) if x % 8 == s2 and x not in (h1, h2)))):
    b.mirrorLog.append((float(t), i, k, 1, r1, True, False))         # three true senders take the lead
b.apply(br, ('share', (k, 1, r1)), 5)                                 # relay 2: a re-relay of the new leader, true; false share 1/2
other = next(k2 for k2 in b.open if b.squadOfTask[k2] == s and k2 not in b.coupled)
b.apply(br, ('open', other), 6)                                       # a role break
R = relayColumns(cfg, b, {})
assert R['arrivalConfirmed'] == 1.0 and R['reRelays'] == 1.0 and R['roleBreaks'] == 1.0
assert abs(R['filterGain'] - ((1.0 + 0.5) / 2 - 0.5)) < 1e-12        # mirror false shares (1, 1/2) minus false relays (1/2)
n = len(b.coupled)
assert abs(R['truthConfirmable'] - 1 / n) < 1e-12 and abs(R['liarConfirmed'] - 1 / n) < 1e-12 and np.isnan(R['coupledContamination'])
# a board without relays or roles: NaN where undefined
R0 = relayColumns(cfg, RelayBoardEnv(1800, 64, guardSlots=G, squads=8, nLiar=16, couple=0.25, mirrorWindow=True, **kw), {})
assert all(np.isnan(R0[c]) for c in ('arrivalConfirmed', 'filterGain', 'reRelays', 'roleBreaks')) and R0['truthConfirmable'] == 0.0
# numpy bridgeFix and mirrorWindow boards (40 turns): every column present and in range
for arm in ('bridgeFix', 'mirrorWindow'):
    row = runC1bCell(c1bCell('t', arm, 1800, runner='numpy'))[0]
    assert all(c in row for c in RELAY_COLS + ('contamination', 'crossContamination', 'bridgeRelays', 'honestPooled'))
    for c in ('arrivalConfirmed', 'coupledContamination', 'truthConfirmable', 'liarConfirmed', 'contamination'):
        assert np.isnan(row[c]) or 0 <= row[c] <= 1, (arm, c, row[c])
    print(arm, {c: round(row[c], 3) for c in RELAY_COLS + ('contamination', 'honestPooled', 'bridgeRelays')})
    assert (row['bridgeRelays'] > 0) == (arm == 'bridgeFix') and (arm != 'bridgeFix' or row['roleBreaks'] == 0)

In [ ]:
# test_relay_off_replay (AC-1, FR-A1): a whole numpy run of C1's bridge1 world forced through RelayBoardEnv (no relay key)
# equals the same run on BridgeBoardEnv: the same row and the same turn log
import explRL.clue_roll as CR, explRL.clue_runs as RU
from explRL.bridge_board import BridgeBoardEnv
c = c1bCell('t', 'bridge1', 1800, runner='numpy')
cfg1 = LLMConfig.fromDict(dict(c['cfg'], turns=12))
r0 = replay(cfg1, 1800)
old = CR.boardClass, RU.boardClass
CR.boardClass = RU.boardClass = lambda w: RelayBoardEnv
try:
    r1 = replay(cfg1, 1800)
finally:
    CR.boardClass, RU.boardClass = old
assert type(r0.board) is BridgeBoardEnv and type(r1.board) is RelayBoardEnv
assert repr(r0.rows) == repr(r1.rows) and repr(r0.log) == repr(r1.log) and r1.board.relayLog == r0.board.relayLog

In [ ]:
#| slow
# test_reused23 (AC-7, FR-C2): C1's 36 reused boards replay from the cache at $0 with every saved column unchanged, and gain
# the relay columns
from explRL.bench_grid import readCSV
from explRL.liar_trace import cacheOnlyClient
from explRL.relay_grid import reusedCells23
from explRL.train import demo_artifact_dir
C1 = demo_artifact_dir('c1', 'multiagent/colony')
CACHE = str(demo_artifact_dir(None, 'multiagent/llm') / 'cache')
if not (C1 / 'runs.csv').exists():
    print("skipped: C1's rows are not here")
else:
    rows = reusedRows23(lambda cfg: cacheOnlyClient(cfg, CACHE), readCSV(C1 / 'runs.csv') + readCSV(C1 / 'reused.csv'), reusedCells23())
    assert len(rows) == 36 and all(c in r for r in rows for c in RELAY_COLS)
    assert all(np.isnan(r['roleBreaks']) for r in rows)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()